In [1]:
!pip install -q ultralytics 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.1 MB/s eta 0:00:00


In [2]:
import os
import json
import shutil
import random
import cv2
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split

In [3]:
# JSON file
json_path = "/kaggle/input/datasets/lsvsandeepm/cotton-reconciled-json/Cotton_reconciled.json"

# Image source folders
image_sources = [
    "/kaggle/input/datasets/lsvsandeepm/cotton-deficiencies",
    "/kaggle/input/datasets/lsvsandeepm/cotton-diseases",
    "/kaggle/input/datasets/lsvsandeepm/cotton-healthy",
    "/kaggle/input/datasets/lsvsandeepm/crop-pests",
]

# Working dataset folder
base_dir = "/kaggle/working/cotton_yolo_dataset"
os.makedirs(base_dir, exist_ok=True)

print("Paths ready")

Paths ready


In [4]:
with open(json_path) as f:
    coco_data = json.load(f)

images = coco_data["images"]
annotations = coco_data["annotations"]
categories = coco_data["categories"]

print("Total Images:", len(images))
print("Total Annotations:", len(annotations))
print("Classes:", [cat["name"] for cat in categories])

Total Images: 2774
Total Annotations: 12295
Classes: ['Aphids', 'Anthracnose', 'Bacterial Blight', 'Grey Mildew and Areolate Mildew', 'Leaf Hopper', 'Mealy Bug', 'Tobacco Caterpillar', 'Whitefly']


In [5]:
for split in ["train", "val"]:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)

print("Folder structure created")

Folder structure created


In [6]:
image_annotations = defaultdict(list)

for ann in annotations:
    image_annotations[ann["image_id"]].append(ann)

print("Annotations grouped")

Annotations grouped


In [7]:
image_ids = [img["id"] for img in images]

train_ids, val_ids = train_test_split(
    image_ids,
    test_size=0.2,
    random_state=42
)

print("Train:", len(train_ids))
print("Val:", len(val_ids))

Train: 2219
Val: 555


In [8]:
def convert_bbox(size, bbox):
    dw = 1. / size[0]
    dh = 1. / size[1]

    x = bbox[0] + bbox[2]/2.0
    y = bbox[1] + bbox[3]/2.0
    w = bbox[2]
    h = bbox[3]

    x *= dw
    w *= dw
    y *= dh
    h *= dh

    return (x, y, w, h)

In [9]:
def find_image_path(base_name):
    for folder in image_sources:
        for root, _, files in os.walk(folder):
            if base_name in files:
                return os.path.join(root, base_name)
    return None


def process_split(split_ids, split_name):
    for img in tqdm(images):
        if img["id"] not in split_ids:
            continue

        file_name = img["file_name"]
        width = img["width"]
        height = img["height"]

        base_name = os.path.basename(file_name)
        src_path = find_image_path(base_name)

        if src_path is None:
            print("Image not found:", base_name)
            continue

        dst_img_path = os.path.join(base_dir, "images", split_name, base_name)
        shutil.copy(src_path, dst_img_path)

        label_name = base_name.rsplit(".", 1)[0] + ".txt"
        label_path = os.path.join(base_dir, "labels", split_name, label_name)

        with open(label_path, "w") as f:
            for ann in image_annotations[img["id"]]:
                bbox = ann["bbox"]
                class_id = ann["category_id"] - 1
                x, y, w, h = convert_bbox((width, height), bbox)
                f.write(f"{class_id} {x} {y} {w} {h}\n")


process_split(train_ids, "train")
process_split(val_ids, "val")

print("YOLO conversion complete")

100%|██████████| 2774/2774 [01:01<00:00, 44.81it/s]

YOLO conversion complete


In [10]:
class_names = [cat["name"] for cat in categories]

with open(f"{base_dir}/data.yaml", "w") as f:
    f.write(f"path: {base_dir}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write(f"nc: {len(class_names)}\n")
    f.write("names:\n")
    for i, name in enumerate(class_names):
        f.write(f"  {i}: {name}\n")

print("data.yaml created")

data.yaml created


In [11]:
shutil.rmtree("/kaggle/working/runs", ignore_errors=True)
print("Old runs cleared")

Old runs cleared


In [12]:
# from ultralytics import YOLO

# model = YOLO("yolov8m.pt")

# model.train(
#     data=f"{base_dir}/data.yaml",
    
#     epochs=180,
#     imgsz=1024,
#     batch=8,
#     device=0,

#     optimizer="AdamW",
#     lr0=0.001,
#     lrf=0.01,
#     cos_lr=True,
#     weight_decay=0.0005,
#     warmup_epochs=5,

#     mosaic=1.0,
#     mixup=0.3,
#     copy_paste=0.2,
#     scale=0.7,
#     degrees=10,
#     translate=0.15,
#     fliplr=0.5,

#     hsv_h=0.02,
#     hsv_s=0.8,
#     hsv_v=0.5,

#     patience=40,
#     cache=False,
#     workers=2,

#     project="cotton_training",
#     name="v2_high_accuracy"
# )



from ultralytics import YOLO

model = YOLO("yolov8m.pt")   # Bigger model

model.train(
    data=f"{base_dir}/data.yaml",

    epochs=10,
    imgsz=640,
    batch=16,
    device=0,

    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    patience=50,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    close_mosaic=10,

    project="cotton_training",
    name="v3_balanced_optimized"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/cotton_yolo_dataset/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, im

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78e82eab0f20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [13]:
source = "/kaggle/working/runs/detect/cotton_training/v2_high_accuracy/weights/best.pt"
destination = "/kaggle/working/cotton_best_model.pt"

if os.path.exists(source):
    shutil.copy(source, destination)
    print("Best model saved")

In [14]:
best_model = YOLO("/kaggle/working/runs/detect/cotton_training/v3_balanced_optimized/weights/best.pt")

metrics = best_model.val()

print("\nFINAL METRICS:")
print("Precision:", metrics.box.p)
print("Recall:", metrics.box.r)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,844,392 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2950.2±1128.2 MB/s, size: 928.0 KB)
val: Scanning /kaggle/working/cotton_yolo_dataset/labels/val.cache... 555 images, 0 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 555/555 258.6Mit/s 0.0s
val: /kaggle/working/cotton_yolo_dataset/images/val/Cotton Anthracnose (110).jpg: 7 duplicate labels removed
val: /kaggle/working/cotton_yolo_dataset/images/val/Cotton Anthracnose (136).jpg: 11 duplicate labels removed
val: /kaggle/working/cotton_yolo_dataset/images/val/Cotton Anthracnose (223).jpg: 10 duplicate labels removed
val: /kaggle/working/cotton_yolo_dataset/images/val/Cotton Anthracnose (23).jpg: corrupt JPEG restored and saved
val: /kaggle/working/cotton_yolo_dataset/images/val/Cotton Anthracnose (26).jpg: corrupt JPEG restored and saved
val: /kaggle/working/cotton_yolo_dataset/imag

In [15]:
# model = YOLO("/kaggle/working/cotton_best_model.pt")

# val_path = f"{base_dir}/images/val"

# all_images = [os.path.join(val_path, f) 
#               for f in os.listdir(val_path) 
#               if f.lower().endswith(('.jpg','.jpeg','.png'))]

# sample_images = random.sample(all_images, 5)

# for img_path in sample_images:
#     print("="*60)
#     print("IMAGE:", os.path.basename(img_path))

#     results = model.predict(
#         source=img_path,
#         imgsz=1024,
#         conf=0.3,
#         iou=0.5,
#         save=False
#     )

#     result = results[0]

#     if result.boxes and len(result.boxes) > 0:
#         for box in result.boxes:
#             cls = int(box.cls[0])
#             conf = float(box.conf[0])
#             print(f"Detected → {model.names[cls]} | Confidence: {conf:.3f}")
#     else:
#         print("No detections")

#     plotted = result.plot()
#     plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)

#     plt.figure(figsize=(8,8))
#     plt.imshow(plotted)
#     plt.axis("off")
#     plt.show()